# RAG demo — legal context for zero-shot crime classification (AirLLM backend)

Pipeline:
1. Fetch **US Code Title 18** statutes from govinfo.gov (no API key).
2. Fetch real **CourtListener** judgments (two-hop: search → opinions/{id}/, token via `.env`).
3. Fetch **UK Acts of Parliament** from legislation.gov.uk (no key, OGL).
4. Merge all three into one **FAISS** index over statutes and case law (`LegalRetriever`, BGE-small).
5. Run `AirLLMClassifier` on `data/sample.csv` **with vs. without** the retriever.
6. Compare macro-F1 and inspect the `reasoning` field — with RAG it should cite USC sections, UK Acts, and/or case names.

**Backend:** [AirLLM](https://github.com/lyogavin/airllm) — loads one transformer layer at a time from disk → runs 7B/70B LLMs on tiny GPUs / Mac Silicon. MLX auto-selected on macOS. Generation is slow (seconds-to-minutes per row).

Prereqs:
- `uv sync && uv sync --extra airllm` (CUDA: `--extra airllm-cuda`; Mac: `--extra airllm-mlx`).
- Copy `.env.example` → `.env` and add `COURTLISTENER_API_TOKEN=...` if you have one.
- Open the notebook with the `crimellm` kernel.
- ~14 GB free disk for Qwen2.5-7B weights.

In [ ]:
import os
from pathlib import Path

from crimellm import (
    AirLLMClassifier,
    LegalRetriever,
    fetch_us_code_sections,
    download_courtlistener,
    load_env,
    load_jsonl,
    load_dataset_from_csv,
)

# Load secrets from `.env` (CourtListener / govinfo / Anthropic tokens).
# Copy `.env.example` to `.env` at the project root and fill in values.
env_path = load_env()
print(f'.env loaded: {env_path}' if env_path else 'no .env found (anon access only)')
print(f"  COURTLISTENER_API_TOKEN: {'set' if os.environ.get('COURTLISTENER_API_TOKEN') else 'unset (anon, 5k/day)'}")
print(f"  GOVINFO_API_KEY       : {'set' if os.environ.get('GOVINFO_API_KEY') else 'unset (DEMO_KEY, 30/hr)'}")

CORPORA = Path('../data/corpora')
CORPORA.mkdir(parents=True, exist_ok=True)
USC_BASE = CORPORA / 'usc18_demo'
CL_BASE = CORPORA / 'cl_demo'
COMBINED_BASE = CORPORA / 'legal_combined'

## 1. Ingest USC Title 18 — curated criminal-law sections

Hand-picked statutes covering the crime types in `sample.csv`. For a full ingest, see `download_us_code(...)` which walks all 1939 granules of Title 18 via api.govinfo.gov (needs a free [api.data.gov key](https://api.data.gov/signup/) — `DEMO_KEY` is rate-limited to 30 req/hour).

In [ ]:
TITLE_18_CRIME_GRANULES = [
    'USCODE-2023-title18-partI-chap51-sec1111',   # § 1111. Murder
    'USCODE-2023-title18-partI-chap51-sec1112',   # § 1112. Manslaughter
    'USCODE-2023-title18-partI-chap103-sec2113',  # § 2113. Bank robbery
    'USCODE-2023-title18-partI-chap31-sec641',    # § 641. Public money/property theft
    'USCODE-2023-title18-partI-chap31-sec659',    # § 659. Interstate-shipment theft
    'USCODE-2023-title18-partI-chap47-sec1001',   # § 1001. False statements
    'USCODE-2023-title18-partI-chap63-sec1341',   # § 1341. Mail fraud
    'USCODE-2023-title18-partI-chap63-sec1343',   # § 1343. Wire fraud
    'USCODE-2023-title18-partI-chap25-sec471',    # § 471. Counterfeiting US obligations
    'USCODE-2023-title18-partI-chap7-sec113',     # § 113. Assault (maritime/territorial)
    'USCODE-2023-title18-partI-chap41-sec875',    # § 875. Interstate threats
]
if not (USC_BASE.parent / (USC_BASE.name + '.jsonl')).exists():
    n = fetch_us_code_sections(USC_BASE, TITLE_18_CRIME_GRANULES, year=2023, title='18')
    print(f'wrote {n} statute records')
else:
    print(f'already exists: {USC_BASE}.jsonl')

## 2. Ingest CourtListener — real case-law judgments

`download_courtlistener` is a **two-hop** pipeline:

1. `/api/rest/v4/search/?type=o&q=<query>` — Elasticsearch over opinions, returns case name, court, citation, dateFiled, opinion id, and a highlighted snippet. Anon-friendly.
2. `/api/rest/v4/opinions/{id}/` — fetches the full `plain_text` body for each hit. **Requires a token** (set `COURTLISTENER_API_TOKEN` in `.env`).

If no token is set, falls back to snippet-only (still useful for retrieval). With token, builds out to 8 KB of body text per case. Includes pacing + 429 retry/backoff — burst limits are real.

Seed query targets the federal-crime domain so the cases overlap our classifier.

In [ ]:
if not (CL_BASE.parent / (CL_BASE.name + '.jsonl')).exists():
    n = download_courtlistener(
        CL_BASE,
        query='bank robbery OR mail fraud OR wire fraud OR burglary OR aggravated assault OR threats OR forgery',
        max_docs=25,
        order_by='score desc',
    )
    print(f'wrote {n} judgment records')
else:
    print(f'already exists: {CL_BASE}.jsonl')

In [ ]:
usc_records = load_jsonl(str(USC_BASE) + '.jsonl')
cl_records = load_jsonl(str(CL_BASE) + '.jsonl')
print(f'{len(usc_records)} statutes  +  {len(cl_records)} judgments  =  {len(usc_records)+len(cl_records)} total\n')
print('--- statutes ---')
for r in usc_records:
    print(f"  {r['citation']:>20}  {r['metadata']['heading'][:60]}")
print('\n--- cases (first 8) ---')
for r in cl_records[:8]:
    md = r['metadata']
    print(f"  {md['date_filed']:<12}  {md['court_id']:<12}  {md['case_name'][:60]}")

## 3. Ingest UK Acts of Parliament — legislation.gov.uk

`download_uk_legislation` pulls whole-act CLML XML from `legislation.gov.uk/{type}/{year}/{number}/data.xml` — Open Government Licence, no key, no rate limit, ~1 HTTP request per Act. Iterates `<P1>` body sections (skips schedules), extracts heading + statutory text.

Default `UK_CRIMINAL_ACTS` covers the main criminal-law statutes (Fraud Act 2006, Theft Act 1968, Misuse of Drugs Act 1971, Criminal Damage Act 1971, Offences against the Person Act 1861, Public Order Act 1986, Sexual Offences Act 2003, Modern Slavery Act 2015, Bribery Act 2010).

In [ ]:
from crimellm import download_uk_legislation, UK_CRIMINAL_ACTS

UK_BASE = CORPORA / 'uk_demo'
if not (UK_BASE.parent / (UK_BASE.name + '.jsonl')).exists():
    n = download_uk_legislation(UK_BASE, statutes=UK_CRIMINAL_ACTS)
    print(f'wrote {n} UK statute sections')
else:
    print(f'already exists: {UK_BASE}.jsonl')

uk_records = load_jsonl(str(UK_BASE) + '.jsonl')
from collections import Counter
print('\nsections per Act:')
for act, c in Counter(r['metadata']['act_title'] for r in uk_records).items():
    print(f'  {c:>3}  {act}')

## 3. Build a combined FAISS index

One vector store over both sources — at retrieval time the LLM sees a mix of statutes and case law, ranked by cosine similarity. `BAAI/bge-small-en-v1.5` (384-dim) handles legal English well enough for small/medium corpora.

In [ ]:
all_records = usc_records + cl_records + uk_records
print(f'{len(usc_records)} USC + {len(cl_records)} CL + {len(uk_records)} UK = {len(all_records)} total')

if not (COMBINED_BASE.parent / (COMBINED_BASE.name + '.faiss')).exists():
    retriever = LegalRetriever.build(all_records, COMBINED_BASE,
                                     embedding_model='BAAI/bge-small-en-v1.5')
else:
    retriever = LegalRetriever.load(COMBINED_BASE)
print(f'index size: {len(retriever)}')

## 4. Retrieval sanity check — mixed sources

Criminal queries should pull the right USC section AND analogous cases; benign queries should score uniformly low.

In [ ]:
for q in [
    'he broke into the shop at night and stole money from the till',
    'she defrauded investors with false statements about company finances',
    'he dishonestly appropriated property belonging to another',
    'they bribed a foreign official to win a contract',
    'the defendant assaulted a federal officer during the arrest',
    'she went for a walk in the park and read a book',
]:
    print(f'> {q}')
    for h in retriever.retrieve(q, k=4):
        label = h.metadata.get('heading') or h.metadata.get('case_name', '')
        print(f'  {h.score:.3f}  [{h.source:>14}]  {h.citation[:30]:<30}  {label[:55]}')
    print()

## 5. Classify with vs. without RAG — AirLLM backend

Same model, same system prompt, only the user message changes (retrieved context block prepended). First run downloads ~14 GB of Qwen2.5-7B-Instruct weights — be patient.

**Generation is slow.** AirLLM streams layers from disk → seconds per token. For a 6-row test set expect ~10-30 minutes on Mac MLX, faster on CUDA with `compression='4bit'`. Lower `max_new_tokens` (default 120) to trade detail for speed.

In [ ]:
sample = Path("../data/sample.csv")
ds = load_dataset_from_csv(path=sample)
texts = list(ds['test']['text'])
labels = list(ds['test']['label'])
print(f'{len(texts)} test rows')

In [ ]:
MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'  # or 'Qwen/Qwen2.5-3B-Instruct' (~6 GB) for faster smoke tests

baseline = AirLLMClassifier(model_id=MODEL_ID)
ragclf = AirLLMClassifier(model_id=MODEL_ID, retriever=retriever)

In [ ]:
results_base = baseline.classify_many(texts)
results_rag = ragclf.classify_many(texts)

In [ ]:
import pandas as pd
from sklearn.metrics import f1_score

LABEL_NAMES = ['no', 'yes', 'unclear']
to_id = lambda s: LABEL_NAMES.index(s) if s in LABEL_NAMES else 2

pred_base = [to_id(r.label) for r in results_base]
pred_rag = [to_id(r.label) for r in results_rag]

print(f'macro-F1 baseline: {f1_score(labels, pred_base, average="macro", zero_division=0):.3f}')
print(f'macro-F1 + RAG  : {f1_score(labels, pred_rag, average="macro", zero_division=0):.3f}')
pd.set_option('display.max_colwidth', None)   # don't truncate cell contents
pd.set_option('display.max_rows', None)       # show all rows
pd.set_option('display.width', None)
df = pd.DataFrame({
    'text': texts,
    'gold': [LABEL_NAMES[i] for i in labels],
    'base_label': [r.label for r in results_base],
    'base_reason': [r.reasoning for r in results_base],
    'rag_label': [r.label for r in results_rag],
    'rag_reason': [r.reasoning for r in results_rag],
})
display(df)